### LLM Chain 만들기


## 1. 환경 구성

### 1) 라이브러리 설치

In [ ]:
# uv add python-dotenv langchain langchain-openai

### 2) GROQ 인증키 설정

In [1]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print(GROQ_API_KEY[:3])

gsk


In [2]:
import langchain
print(langchain.__version__)

1.4.3


In [4]:
from langchain_openai import ChatOpenAI

# Groq(OpenAI 호환) 공통 설정 — 모델/온도 변경은 이곳만 수정
GROQ_BASE_URL = "https://api.groq.com/openai/v1"  # Groq API 엔드포인트
GROQ_MODEL = "openai/gpt-oss-120b"  # 대안: "qwen/qwen3.8-27b"
#MOVIE_MODEL = "qwen/qwen3.8-27b"  # 영화 추천 예제용 
TEMPERATURE = 0.7


def get_llm(model: str = GROQ_MODEL, temperature: float = TEMPERATURE) -> ChatOpenAI:
    """Groq 엔드포인트를 사용하는 ChatOpenAI 인스턴스를 생성합니다."""
    return ChatOpenAI(
        api_key=GROQ_API_KEY,
        base_url=GROQ_BASE_URL,
        model=model,
        temperature=temperature,
    )


# 기본 llm (이후 셀에서 재사용)
llm = get_llm()
print(f'model name = {llm.model_name}')

model name = openai/gpt-oss-120b


## 2. LLM Chain

### 1) Prompt + LLM


In [ ]:
# model: 공통 설정 셀의 llm 사용

# chain 실행
result = llm.invoke("인공지능 모델의 학습 원리에 대하여 쉽게 설명해 주세요.")
print(type(result))
print(result)

In [ ]:
print(result.content)

### 2) PromptTemplate + LLM

In [6]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("You are an expert in AI Expert. Answer the question.\
                                       <Question>: {input}에 대해 쉽게 설명해주세요.")

print(type(prompt))
print(prompt)

<class 'langchain_core.prompts.prompt.PromptTemplate'>
input_variables=['input'] input_types={} partial_variables={} template='You are an expert in AI Expert. Answer the question.                                       <Question>: {input}에 대해 쉽게 설명해주세요.'


In [7]:
# llm: 공통 설정 셀의 llm 사용
# chain 연결 (LCEL)
chain = prompt | llm
print(type(chain))

# chain 호출
result = chain.invoke({"input": "인공지능 모델의 학습 원리"})
print(type(result))
print(result)

<class 'langchain_core.runnables.base.RunnableSequence'>
<class 'langchain_core.messages.ai.AIMessage'>
content='## 인공지능 모델이 어떻게 배우는지 쉽게 이해하기\n\n### 1. “학생”과 “문제집”을 떠올려 보세요\n- **학생** : 인공지능 모델(예: 이미지 인식 모델, 번역 모델 등)  \n- **문제집** : 우리가 모델에게 보여주는 **데이터**(사진, 문장, 숫자 등)와 **정답**(라벨)\n\n학생이 문제집을 풀면서 점점 실력이 늘어나듯, 모델도 데이터를 보면서 스스로 규칙을 찾아갑니다.\n\n---\n\n## 2. 학습 과정의 핵심 3단계\n\n| 단계 | 뭐 하는 일? | 비유 |\n|------|-------------|------|\n| **① 데이터 입력** | 모델에 **입력값**(예: 사진)과 **정답**(예: ‘고양이’)을 줍니다. | 교과서와 정답지를 동시에 보는 것 |\n| **② 예측 & 오류 측정** | 모델이 현재 배운 지식으로 **예측**을 하고, 예측이 정답과 얼마나 다른지 **오류(손실, loss)** 를 계산합니다. | 시험을 보고 점수를 매기는 순간 |\n| **③ 피드백(학습) → 업데이트** | 오류를 줄이기 위해 **가중치**(모델 안의 숫자들)를 조금씩 바꿉니다. 이때 주로 **경사하강법(gradient descent)** 이라는 방법을 씁니다. | 틀린 문제를 다시 공부해서 머릿속 기억을 고쳐 넣는 과정 |\n\n이 과정을 **많은 데이터**와 **많은 반복(에포크)** 에 걸쳐 수행하면, 모델은 점점 더 정확한 예측을 할 수 있게 됩니다.\n\n---\n\n## 3. 핵심 개념을 일상적인 비유로 풀어보기\n\n### 1) 가중치 = “뇌 속의 스위치”\n- 모델 안에는 수천·수백만 개의 **가중치**가 있어요. 각각은 “이 정보가 얼마나 중요한가?”를 조절하는 스위치와 비슷합니다.\n- 학습이 진행되면, 중요한 

In [8]:
print(result.content)

## 인공지능 모델이 어떻게 배우는지 쉽게 이해하기

### 1. “학생”과 “문제집”을 떠올려 보세요
- **학생** : 인공지능 모델(예: 이미지 인식 모델, 번역 모델 등)  
- **문제집** : 우리가 모델에게 보여주는 **데이터**(사진, 문장, 숫자 등)와 **정답**(라벨)

학생이 문제집을 풀면서 점점 실력이 늘어나듯, 모델도 데이터를 보면서 스스로 규칙을 찾아갑니다.

---

## 2. 학습 과정의 핵심 3단계

| 단계 | 뭐 하는 일? | 비유 |
|------|-------------|------|
| **① 데이터 입력** | 모델에 **입력값**(예: 사진)과 **정답**(예: ‘고양이’)을 줍니다. | 교과서와 정답지를 동시에 보는 것 |
| **② 예측 & 오류 측정** | 모델이 현재 배운 지식으로 **예측**을 하고, 예측이 정답과 얼마나 다른지 **오류(손실, loss)** 를 계산합니다. | 시험을 보고 점수를 매기는 순간 |
| **③ 피드백(학습) → 업데이트** | 오류를 줄이기 위해 **가중치**(모델 안의 숫자들)를 조금씩 바꿉니다. 이때 주로 **경사하강법(gradient descent)** 이라는 방법을 씁니다. | 틀린 문제를 다시 공부해서 머릿속 기억을 고쳐 넣는 과정 |

이 과정을 **많은 데이터**와 **많은 반복(에포크)** 에 걸쳐 수행하면, 모델은 점점 더 정확한 예측을 할 수 있게 됩니다.

---

## 3. 핵심 개념을 일상적인 비유로 풀어보기

### 1) 가중치 = “뇌 속의 스위치”
- 모델 안에는 수천·수백만 개의 **가중치**가 있어요. 각각은 “이 정보가 얼마나 중요한가?”를 조절하는 스위치와 비슷합니다.
- 학습이 진행되면, 중요한 스위치는 **켜지고** 덜 중요한 스위치는 **꺼집니다**.

### 2) 손실 함수 = “점수표”
- 손실 함수는 “얼마나 틀렸는가?”를 숫자로 나타낸 것이에요.
- 점수가 낮을수록(손실이 작을수록) 모델이 잘 배운 거예요.

### 3) 경사하강

### 3) PromptTemplate + LLM(invoke()) + StrOutputParser

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 컴포넌트 정의 (llm 은 공통 설정 셀의 llm 사용)
prompt = PromptTemplate.from_template("You are an expert in AI Expert. \
                                      Answer the question. <Question>: {input}에 대해 쉽게 설명해주세요.")

output_parser = StrOutputParser()

# 2. chain 생성 (LCEL)
chain = prompt | llm | output_parser
print(type(chain))

# 3. chain의 invoke 호출
result = chain.invoke({"input": "인공지능 모델의 학습 원리"})
print(type(result))
print(result)

### 4) PromptTemplate + LLM(stream()) + StrOutputParser

In [10]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 컴포넌트 정의 (llm 은 공통 설정 셀의 llm 사용)
prompt = PromptTemplate.from_template("You are an expert in AI Expert. \
                                      Answer the question. <Question>: {input}에 대해 쉽게 설명해주세요.")

# chain 연결 (LCEL)
chain = prompt | llm | StrOutputParser()

# 스트리밍 출력을 위한 요청
answer = chain.stream({"input": "인공지능 모델의 학습 원리"})
# 스트리밍 출력
#print(answer)

for token in answer:
    # 스트림에서 받은 데이터의 내용을 출력합니다. 줄바꿈 없이 이어서 출력하고, 버퍼를 즉시 비웁니다.
    print(token, end="", flush=True)

### 인공지능 모델, 특히 **머신러닝·딥러닝 모델**이 어떻게 학습하는지 쉽게 풀어 설명해 볼게요.

---

## 1. “뇌가 배우는” 과정과 비슷하게 생각하면 됩니다

| 사람(뇌) | 인공지능 모델 |
|----------|----------------|
| **경험(데이터)**을 통해 **패턴**을 찾는다 | **학습 데이터**를 보며 **규칙(가중치)**을 찾아낸다 |
| **오답**을 하면 **피드백**을 받아 고친다 | **손실(Loss)** 값을 계산하고 **오차 역전파**로 고친다 |
| **반복**해서 점점 더 정확해진다 | **에폭(Epoch)**·**반복**을 통해 점점 성능이 올라간다 |

---

## 2. 학습 과정의 핵심 단계 4가지

1. **데이터 준비**  
   - **입력**(예: 사진, 텍스트, 소리) + **정답 라벨**(예: “고양이”, “스팸 메일”)을 쌍으로 만든 데이터셋을 준비합니다.  
   - 데이터가 많고 다양할수록 모델이 잘 배웁니다.

2. **예측(Forward Pass)**  
   - 모델은 현재 가지고 있는 **가중치(파라미터)**를 이용해 입력을 처리하고 **예측값**을 냅니다.  
   - 예를 들어, 사진을 넣으면 “고양이일 확률 0.73” 같은 값을 출력합니다.

3. **오차 계산(손실 함수)**  
   - 모델이 낸 예측값과 정답 라벨 사이의 차이를 **손실(Loss)**이라는 수치로 나타냅니다.  
   - 손실이 작을수록 모델이 정답에 가깝게 예측하고 있다는 뜻입니다.  
   - 흔히 쓰는 손실 함수: **MSE(평균 제곱 오차)**, **Cross‑Entropy(교차 엔트로피)** 등.

4. **가중치 업데이트(Backpropagation + Optimizer)**  
   - 손실을 줄이기 위해 **가중치를 어떻게 바꿔야 할까?**를 계산합니다.  
   - **미분**을 이용해 “손실이 가장 크게 줄어드는 방향”을 찾고, 그 방향으로 조금씩 움직입니다.  
   - 이

##### 2) Multiple Chains
* Multi Chain을 활용한 영화 추천 및 줄거리 요약 

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
prompt1 = ChatPromptTemplate.from_template("{genre} 장르에서 추천할 만한 영화를 한 편 알려주세요.")

# Step 2: 추천된 영화의 줄거리를 요약
prompt2 = ChatPromptTemplate.from_template("{movie} 추천한 영화의 제목을 먼저 알려주시고,\
                                            줄을 바꾸어서 영화의 정보(제목,감독,캐스팅,줄거리)를 알려 주세요")

# 영화 예제용 모델 (공통 설정 셀의 get_llm 사용)
llm = get_llm(GROQ_MODEL)

# 체인 1: 영화 추천 (입력: 장르 → 출력: 영화 제목)
chain1 = prompt1 | llm | StrOutputParser()

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
movie = chain1.invoke({"genre": "Drama"})  # 영화 제목 얻기

print(type(movie))
print(" ===> 추천된 영화:")  # movie 값 출력
print(movie)


<class 'langchain_core.messages.base.TextAccessor'>
 ===> 추천된 영화:
**추천 영화: <그린 북 (Green Book, 2018) — 감독: 피터 패럴리**  

### 간단 소개
- **장르**: 드라마 / 코미디
- **주연**: 마하샬라 알리, 비고 모텐센
- **줄거리**: 1960년대 초, 흑인 피아니스트 돈 셜리(마하샬라 알리)와 그의 백인 운전사 토니 리프(비고 모텐센)가 미국 남부를 순회하며 콘서트를 진행한다. 인종 차별이 일상화된 시대에 두 사람은 서로 다른 배경과 편견을 극복하고 진정한 우정을 쌓아가게 된다.  

### 왜 추천하는가?
1. **감동적인 인간 관계** – 서로 다른 세계에 살던 두 주인공이 함께 겪는 갈등과 화해 과정을 통해 “우정”과 “이해”의 가치를 깊이 있게 그린다.  
2. **역사적 배경** – 1960년대 미국 남부의 인종 차별 현실을 사실적으로 보여주면서도, 오늘날 우리 사회에서도 공감할 수 있는 메시지를 전달한다.  
3. **수상 경력** – 아카데미 시상식에서 최우수 작품상, 최우수 감독상, 최우수 각본상 등 3관왕을 차지해 품질이 입증된 작품이다.  
4. **밝은 톤과 유머** – 무거운 주제를 다루면서도 적절한 코미디 요소가 섞여 있어 관객이 지루함 없이 몰입할 수 있다.  

### 감상 포인트
- **음악**: 클래식 피아노 연주와 재즈 사운드트랙이 영화 분위기를 한층 고조시킨다.  
- **연기**: 비고 모텐센의 코믹하면서도 인간적인 연기와 마하샬라 알리의 섬세한 감정 표현이 뛰어나다.  
- **시각적 연출**: 남부의 풍경과 60년대 인테리어, 의상 등 시대적 디테일이 풍부하게 재현돼 몰입감을 높인다.  

### 시청 방법
- **스트리밍**: 넷플릭스, 아마존 프라임 비디오, 왓챠 등에서 제공 중(지역에 따라 차이가 있을 수 있음).  
- **구매/대여**: 구글 플레이 영화, Apple TV, YouTube Movies 등에서도 디지털 

In [12]:
# 체인 2: 줄거리 요약 (입력: 영화 제목 → 출력: 줄거리)
chain2 = (
    {"movie": chain1}  # chain1의 출력을 movie 변수로 전달
    | prompt2
    | llm
    | StrOutputParser()
)
print(chain2)

# 실행: "SF" 장르의 영화 추천 및 줄거리 요약
response = chain2.invoke({"genre": "Drama"})
print("\n🔹 영화 줄거리 요약:\n", response) 

first={
  movie: ChatPromptTemplate(input_variables=['genre'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['genre'], input_types={}, partial_variables={}, template='{genre} 장르에서 추천할 만한 영화를 한 편 알려주세요.'), additional_kwargs={})])
         | ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3', 'langchain-openai': '1.6.7'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002DD79936450>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002DD79048860>, root_client=<openai.OpenAI object at 0x000002DD5FD7CC80>, root_async_client=<openai.AsyncOpenAI object at 0x000002DD79937500>, model_name='openai/gpt-oss-120b', temperature=0.7, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.groq.com/openai/v1', stream_chunk_timeout=120.0)
         | StrOutputParser()
} middle=[ChatPromptTemplate

##### chain1과 chain2에서 영화 제목이 일관되게 전달 되도록 변경 

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from pprint import pprint

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
prompt1 = ChatPromptTemplate.from_template("{genre} 장르에서 추천할 만한 영화를 한 편 알려주세요.")

# Step 2: 추천된 영화의 줄거리를 요약
prompt2 = ChatPromptTemplate.from_template("{movie} 추천한 영화의 제목을 먼저 알려주시고, 줄을 바꾸어서 영화의 정보(제목,감독,캐스팅,줄거리)를 알려 주세요.")

llm = get_llm(GROQ_MODEL)

# 체인 1: 영화 추천 (입력: 장르 → 출력: 영화 제목)
chain1 = prompt1 | llm | StrOutputParser()

# 체인 2: 줄거리 요약 (입력: 영화 제목 → 출력: 줄거리)
chain2 = (
    {"movie": chain1}  # chain1의 출력을 movie 변수로 전달
    | prompt2
    | llm
    | StrOutputParser()
)

# 실행: "Drama" 장르의 영화 추천 및 줄거리 요약
response = chain2.invoke({"genre": "Drama"})
print("\n🔹 영화 줄거리 요약:")
pprint(response)


🔹 영화 줄거리 요약:
('그린 마일 (The Green Mile, 1999)  \n'
 '\n'
 '제목: 그린 마일 (The Green Mile, 1999)  \n'
 '감독: 프랭크 다라본트  \n'
 '캐스팅: 톰 행크스, 마이클 클라크 딘, 데이비드 모스 등  \n'
 '줄거리: 1930년대 미국 남부 교도소에서 사형수 담당관(톰 행크스)과, 신비한 치유 능력을 가진 거대한 몸집의 수감자(마이클 클라크 '
 '딘) 사이에 일어나는 인간성과 용서, 정의에 대한 감동적인 이야기를 그린다.')


In [14]:
# 모델 설정 (공통 설정 셀의 get_llm 사용)
model = get_llm(GROQ_MODEL)

# 멀티 체인 실행 함수
def movie_chain(genre):
    print(f"\n🔹 1단계: '{genre}' 장르 영화 추천")
    
    # 체인 1: 영화 추천
    response1 = model.invoke(f"{genre} 장르에서 유명한 영화 제목 하나만 알려줘")
    movie = response1.content.strip()
    print(f"   → 추천 영화: {movie}")
    
    print(f"\n 2단계: '{movie}' 영화 정보 조회")
    
    # 체인 2: 영화 정보
    response2 = model.invoke(f"{movie} 영화에 대해 간단히 알려줘. 줄거리, 감독, 출연진 포함해서")
    info = response2.content
    print(f"   → 영화 정보:\n{info}")
    
    return movie, info

# 실행
genre = input("원하는 장르를 입력하세요: ")
movie, info = movie_chain(genre)
print(f"\n 최종 결과: '{movie}' 영화가 추천되었습니다!")


🔹 1단계: '드라마' 장르 영화 추천
   → 추천 영화: **The Shawshank Redemption** (쇼생크 탈출) – 드라마 장르의 대표적인 명작 영화 중 하나입니다.

 2단계: '**The Shawshank Redemption** (쇼생크 탈출) – 드라마 장르의 대표적인 명작 영화 중 하나입니다.' 영화 정보 조회
   → 영화 정보:
**영화 정보**  
- **제목**: 쇼생크 탈출 (The Shawshank Redemption)  
- **개봉**: 1994년 (미국)  
- **장르**: 드라마 / 범죄  
- **감독**: 프랭크 다라본트 (Frank Darabont)  
- **원작**: 스티븐 킹(Stephen King)의 단편소설 *“Rita Hayworth and Shawshank Redemption”*  

**주요 출연진**  
| 배우 | 역할 |
|------|------|
| 팀 로빈스 (Tim Robbins) | 앤디 듀프레인 (Andy Dufresne) – 은행가이자 주인공 |
| 모건 프리먼 (Morgan Freeman) | 엘리스 “레드” 레딩 (Ellis “Red” Redding) – 장기 수감자, 내레이터 |
| 밥 건턴 (Bob Gunton) | 노튼 교도소장 (Warden Samuel Norton) |
| 윌리엄 새들러 (William Sadler) | 헤이우드 (Heywood) – 레드와 친한 동료 수감자 |
| 클랜시 브라운 (Clancy Brown) | 하들리 교도관 (Captain Byron Hadley) |
| 길 벨로스 (Gil Bellows) | 토미 (Tommy) – 젊은 수감자 |
| 제임스 휘트모어 (James Whitmore) | 브룩스 (Brooks Hatlen) – 은퇴한 도서관 사서 |

**줄거리 (간략 요약)**  
1. **시작** – 1947년, 은행가 앤디 듀프레인은 아내와 그녀의 연인 살해 혐의로 무기징역을 선고받고 메인 주의 쇼생크 교도소에 수감된다. 그는 실제로